# 04 - Train/Test Split and Linear Regression with OLS

**Objective:** Split the data correctly, then fit Linear Regression with Ordinary Least Squares: first a one-feature baseline, then a multiple-feature model. Interpret coefficients and check generalisation.

**Role in the ML workflow:** *Machine Learning* - learn the relationship between input features and insurance charges.

### Concepts to understand first
* **`X` and `y`:** `X` is the table of input features (age, sex, bmi, children, smoker, region). `y` is the answer we want to predict (`charges`).
* **Why split the data?** We want to know how the model performs on people it has **never seen**. If we measured accuracy on the same rows used for learning, a model that merely memorised them would look perfect. So we hide 20% of the rows (the **test set**) and only look at them for a final check.
* **Why not evaluate only on training data?** Training error is optimistic - it measures memorisation + learning together. Test error measures generalisation.
* **Why must the test set stay unseen during development?** Every time you tune something based on test results, information leaks and the test score becomes over-optimistic. We make all choices with cross-validation on the training set.
* **Linear regression:** predicts `charges` as a weighted sum of features plus a constant.
  * One feature: `predicted_charges = w * age + b` (`w` = weight/slope = extra charges per additional year; `b` = intercept = predicted charges when age = 0, which is an extrapolation here as the youngest person is 18).
  * Several features: `predicted_charges = w1*age + w2*bmi + w3*children + w4*smoker_yes + ... + b`. Each weight is the effect of *its* feature while the others are held fixed.
* **Ordinary Least Squares (OLS):** chooses the weights that minimise the sum of squared residuals. It has a closed-form solution, so no iteration or learning rate is needed.
* **Residual:** `actual charges - predicted charges`. Positive = model under-predicted; negative = over-predicted.

## Setup

In [1]:
import sys
import warnings
from pathlib import Path

# Make the project root importable so we can use the code in src/.
# (Jupyter's working directory is notebooks/, so the project root is one level up.)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

from src.data_preprocessing import load_clean_data, split_data
from src.evaluate import regression_metrics, evaluate_model
from src.train import build_age_only_model, build_ols_model

## Part A - The train/test split

In [3]:
df = load_clean_data()

# split_data does: X = features, y = charges, then
# train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = split_data(df)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)
print(f"Test share: {len(X_test) / len(df):.1%}")

X_train: (1069, 6) | y_train: (1069,)
X_test : (268, 6) | y_test : (268,)
Test share: 20.0%


* `test_size=0.2` -> 20% test, 80% training. `random_state=42` -> the same shuffle every run, so results are reproducible and *every model in this project uses exactly the same rows*, which makes comparisons fair.
* We split **before** fitting any scaler/encoder. The pipelines below learn their parameters from `X_train` only.

In [4]:
X_train.head()

,age,sex,bmi,children,smoker,region
1113,23,male,24.510,0,no,northeast
967,21,male,25.745,2,no,northeast
598,52,female,37.525,2,no,northwest
170,63,male,41.470,0,no,southeast
275,47,female,26.600,2,no,northeast


## Part B - Baseline: age-only OLS

`predicted_charges = w * age + b`

In [5]:
age_model = build_age_only_model().fit(X_train, y_train)

w = age_model.named_steps["model"].coef_[0]
b = age_model.named_steps["model"].intercept_
print(f"w (weight for age) = {w:,.2f}   -> each extra year of age adds about {w:,.0f} to predicted charges")
print(f"b (intercept)      = {b:,.2f}")

# Verify the formula by hand for one person
person = X_test.iloc[[0]]
by_hand = w * person["age"].iloc[0] + b
print(f"\nAge {person['age'].iloc[0]}: by hand = {by_hand:,.2f} | model.predict = {age_model.predict(person)[0]:,.2f}")

w (weight for age) = 242.26   -> each extra year of age adds about 242 to predicted charges
b (intercept)      = 3,534.09

Age 49: by hand = 15,404.74 | model.predict = 15,404.74


## Part C - Multiple linear regression (all six features)

We use the pipeline from `src/train.py`. For OLS we leave numeric columns unscaled so the coefficients stay in readable units (scaling does not change OLS predictions).

In [6]:
ols_model = build_ols_model().fit(X_train, y_train)
print(ols_model)

Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('add_features',
                                  FunctionTransformer(func=<function add_engineered_features at 0x7f907ccdcf40>,
                                                      kw_args={'features': ()})),
                                 ('encode_scale',
                                  ColumnTransformer(transformers=[('numeric',
                                                                   'passthrough',
                                                                   ['age',
                                                                    'bmi',
                                                                    'children']),
                                                                  ('binary',
                                                                   OneHotEncoder(drop='if_binary'),
                                                                   ['sex',
                              

### Coefficients and intercept (`coef_`, `intercept_`)

In [7]:
lin = ols_model.named_steps["model"]
names = ols_model.named_steps["preprocess"].named_steps["encode_scale"].get_feature_names_out()
coef_table = pd.DataFrame({"feature": names, "coefficient": lin.coef_}).set_index("feature")
print(f"intercept_ = {lin.intercept_:,.2f}")
coef_table.round(2)

intercept_ = -11,092.65


,coefficient
feature,
numeric__age,248.21
numeric__bmi,318.70
numeric__children,533.01
binary__sex_male,-101.54
binary__smoker_yes,23077.76
multi__region_northwest,-391.76
multi__region_southeast,-838.92
multi__region_southwest,-659.14


**Interpreting the coefficients** (each is the change in predicted charges for a one-unit increase, *holding the other features fixed*):

* `age` about **+248** per year; `bmi` about **+319** per BMI point; `children` about **+533** per dependant.
* `smoker_yes` about **+23,078** - by far the largest weight: a smoker is predicted to cost about 23,000 more than an otherwise identical non-smoker.
* `sex_male` about -102 and the three region dummies (about -392, -839, -659 relative to the reference region *northeast*) are small compared with the noise in the data. This agrees with the EDA and the CV encoding test: sex and region add little.
* `intercept_` about **-11,093**: the prediction for a hypothetical person with age 0, BMI 0, non-smoker, northeast, 0 children. That person does not exist, so the intercept is just the baseline the other terms are added to - it is not meant to be interpreted on its own, and a negative value is not a problem.

*Limits of these readings:* the model assumes every effect is additive and the same for everyone. The single BMI weight is therefore an average of a strong effect for smokers and a negligible one for non-smokers, which is the weakness the interaction features in notebook 03 address. Coefficients describe associations in this sample, not proven causal effects.

### How a prediction is built (by hand)

In [8]:
encoded_row = ols_model.named_steps["preprocess"].transform(X_test.iloc[[0]])
manual = lin.intercept_ + np.dot(encoded_row[0], lin.coef_)
print(X_test.iloc[0].to_dict())
print(f"\nintercept + sum(weight * value) = {manual:,.2f}")
print(f"ols_model.predict               = {ols_model.predict(X_test.iloc[[0]])[0]:,.2f}")
print(f"actual charges                  = {y_test.iloc[0]:,.2f}")
print(f"residual (actual - predicted)   = {y_test.iloc[0] - manual:,.2f}")

{'age': 49, 'sex': 'male', 'bmi': 22.515, 'children': 0, 'smoker': 'no', 'region': 'northeast'}

intercept + sum(weight * value) = 8,143.69
ols_model.predict               = 8,143.69
actual charges                  = 8,688.86
residual (actual - predicted)   = 545.16


## Part D - Predictions on training and testing data

The coefficients are learned **once**, from the training data. After that they are fixed numbers. Predicting a training row or a test row applies the *same* weights and intercept to that row's feature values - nothing is re-learned. What differs is the quality: training rows were used to choose the weights, test rows were not.

In [9]:
coef_before = lin.coef_.copy()          # snapshot of the learned weights
train_pred = ols_model.predict(X_train)
test_pred  = ols_model.predict(X_test)
print("Weights unchanged after predicting train AND test data:",
      np.array_equal(coef_before, lin.coef_))

residuals_test = y_test - test_pred
pd.DataFrame({"actual": y_test.values[:5], "predicted": test_pred[:5], "residual": residuals_test.values[:5]}).round(2)

Weights unchanged after predicting train AND test data: True


,actual,predicted,residual
0,8688.86,8143.69,545.16
1,5708.87,5737.12,-28.25
2,11436.74,14369.31,-2932.58
3,38746.36,31745.51,7000.84
4,4463.21,8962.39,-4499.18


## Part E - Performance: age-only vs multiple-feature OLS

In [10]:
results = pd.concat([
    evaluate_model("Age-only OLS", age_model, X_train, y_train, X_test, y_test),
    evaluate_model("OLS (6 features)", ols_model, X_train, y_train, X_test, y_test),
], ignore_index=True)
results.round(4)

,Model,Dataset,RMSE,MAE,R2
0,Age-only OLS,Training,11199.3166,8767.9593,0.0839
1,Age-only OLS,Testing,12894.7799,9657.7893,0.0951
2,OLS (6 features),Training,6081.1069,4181.9015,0.7299
3,OLS (6 features),Testing,5956.3429,4177.0456,0.8069


**How to read it** (RMSE and MAE are in charge units - lower is better; R-squared closer to 1 is better; full definitions in notebook 06):

* **Age-only baseline:** test RMSE about 12,895, MAE about 9,658, R-squared about 0.10 - age alone explains only around 10% of the variation in charges. Training and test errors are both large, so this model is **underfitting** (too simple), not overfitting.
* **Multiple-feature OLS:** training RMSE about 6,081 / test RMSE about 5,956; MAE about 4,182 / 4,177; R-squared 0.730 (training) / 0.807 (test). Adding smoker, BMI and the other features cuts the test RMSE by more than half.
* **Training vs test:** the test scores are slightly *better* than the training scores. That is possible and not a bug: the test set has only 268 rows, so its score is noisy, and this particular test sample happens to be a little easier. The 5-fold CV estimate on the training data (R-squared about 0.72, RMSE about 6,124) is a more stable guide to expected performance. The important observation is that training and test errors are close, so the model is **not overfitting**; its limitation is bias (it is too simple to capture the smoker interactions).

## Summary

* Split once (80/20, seed 42); all models share it; transformations are fitted on training data only.
* OLS has a closed-form solution; its coefficients are fixed after fitting and are shared by training and test predictions.
* The multiple-feature model is a large improvement over the age-only baseline, but its residuals suggest a linear model without interactions is still missing the smoker structure found in EDA (tested in notebooks 03 and 06).

## Common errors and fixes
| Error | Cause | Fix |
|---|---|---|
| `NotFittedError` | calling `predict` before `fit` | call `.fit(X_train, y_train)` first |
| `ValueError: could not convert string to float` | raw text columns given to a bare `LinearRegression` | use the pipeline (it encodes first) |
| Different numbers than expected | different seed or no duplicate removal | use `split_data` (seed 42, cleaned data) |
| Fitting on the test set | `fit(X_test, ...)` by mistake | only `fit` on training data; only `predict` on test data |

## Practical task
1. Change `test_size` to 0.3 in a scratch cell and see how metrics move. Why do they change a little?
2. Predict the charge for a 30-year-old non-smoking woman with BMI 25, no children, northeast, using `ols_model.predict` on a one-row DataFrame.
3. Explain in your own words what the `smoker_yes` coefficient means.

**Next:** notebook 05 - Linear regression with Stochastic Gradient Descent.